# Continual Learning: Teaching Models to Learn Without Forgetting

**What we'll learn:**
- The fundamental problem of **catastrophic forgetting** in neural networks
- Why continual learning matters for production AI systems and lifelong learning agents
- Three major approaches: **replay methods**, **regularization methods**, and **parameter isolation**
- Practical implementations of Elastic Weight Consolidation (EWC) and experience replay
- How to evaluate continual learning systems with proper metrics

**Why it matters:**

Most neural networks are trained once on a fixed dataset. But real-world AI systems need to learn continuously:
- A recommendation system must adapt to new products without retraining from scratch
- A language model should learn new facts without forgetting old ones
- A robot must acquire new skills while retaining previous capabilities

The core challenge: when neural networks learn new tasks, they catastrophically forget old ones. We'll explore why this happens and how to solve it.

## 1. Setup

We'll import the necessary libraries and configure our environment for reproducibility.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from copy import deepcopy

from aiml_notebooks import set_seed, get_device, create_dataloaders

%load_ext autoreload
%autoreload 2

Set the random seed for reproducibility across all libraries.

In [ ]:
set_seed(42)

Configure the device (GPU if available, otherwise CPU).

In [ ]:
device = get_device()
print(f"Using device: {device}")

## 2. The Continual Learning Problem

### What is Continual Learning?

**Continual learning** (also called lifelong learning or incremental learning) is the ability to learn new tasks sequentially while maintaining performance on previously learned tasks.

**The challenge:** Standard neural networks suffer from **catastrophic forgetting** — when trained on new tasks, they rapidly forget old knowledge.

**Why does this happen?** Neural networks learn by adjusting weights to minimize loss on the current task. These weight changes often overwrite information needed for previous tasks, especially when:
- Tasks share parameters (which they must for efficiency)
- No data from previous tasks is available during new task training

### Creating Sequential Tasks from MNIST

We'll split MNIST into 5 sequential binary classification tasks:
- Task 0: Classify digits 0 vs 1
- Task 1: Classify digits 2 vs 3
- Task 2: Classify digits 4 vs 5
- Task 3: Classify digits 6 vs 7
- Task 4: Classify digits 8 vs 9

This setup lets us study forgetting in a controlled environment.

In [ ]:
from torchvision import datasets, transforms

# Download MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = datasets.MNIST('data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('data', train=False, download=True, transform=transform)

print(f"Training samples: {len(mnist_train)}")
print(f"Test samples: {len(mnist_test)}")

Now we'll create task-specific datasets by filtering MNIST for specific digit pairs.

In [ ]:
def create_task_datasets(dataset, digit_pairs):
    """
    Create task-specific datasets for sequential learning.
    
    Args:
        dataset: Original MNIST dataset
        digit_pairs: List of (digit_a, digit_b) tuples
    
    Returns:
        List of filtered datasets, one per task
    """
    task_datasets = []
    
    for digit_a, digit_b in digit_pairs:
        # Find indices for this digit pair
        indices = [i for i, (_, label) in enumerate(dataset) 
                   if label == digit_a or label == digit_b]
        
        # Create subset
        subset = torch.utils.data.Subset(dataset, indices)
        task_datasets.append(subset)
    
    return task_datasets

# Define 5 binary classification tasks
digit_pairs = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

train_tasks = create_task_datasets(mnist_train, digit_pairs)
test_tasks = create_task_datasets(mnist_test, digit_pairs)

print(f"Created {len(train_tasks)} tasks")
for i, task in enumerate(train_tasks):
    print(f"Task {i} ({digit_pairs[i]}): {len(task)} training samples")

Let's visualize samples from each task to understand what we're working with.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for task_id, (task, pair) in enumerate(zip(train_tasks, digit_pairs)):
    # Get first sample from task
    img, label = task[0]
    
    axes[task_id].imshow(img.squeeze(), cmap='gray')
    axes[task_id].set_title(f'Task {task_id}\n({pair[0]} vs {pair[1]})')
    axes[task_id].axis('off')

plt.tight_layout()
plt.show()

## 3. A Simple Neural Network for Binary Classification

We'll use a simple feedforward network to demonstrate continual learning concepts. This architecture will be reused across all methods.

In [ ]:
class SimpleNet(nn.Module):
    """Simple feedforward network for MNIST binary classification."""
    
    def __init__(self, hidden_size=400):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 10)  # 10 outputs for all digits
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Test the model
model = SimpleNet().to(device)
test_input = torch.randn(4, 1, 28, 28).to(device)
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### Training and Evaluation Functions

We'll create reusable functions to train on a single task and evaluate across all tasks. This will help us measure forgetting.

In [ ]:
def train_task(model, task_dataset, epochs=3, batch_size=128, lr=0.001):
    """
    Train model on a single task.
    
    Args:
        model: Neural network to train
        task_dataset: Dataset for this task
        epochs: Number of training epochs
        batch_size: Batch size
        lr: Learning rate
    
    Returns:
        List of losses per epoch
    """
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(task_dataset, batch_size=batch_size, shuffle=True)
    
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
    
    return losses

def evaluate_task(model, task_dataset, batch_size=128):
    """
    Evaluate model on a single task.
    
    Returns:
        Accuracy on the task
    """
    model.eval()
    loader = torch.utils.data.DataLoader(task_dataset, batch_size=batch_size, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return 100.0 * correct / total

print("Training and evaluation functions ready")

## 4. Demonstrating Catastrophic Forgetting

### The Baseline: Fine-tuning Without Protection

Let's train our model on tasks sequentially using standard fine-tuning. We'll track accuracy on all tasks after learning each new task.

**Hypothesis:** Performance on early tasks will collapse as we learn new ones.

In [ ]:
# Initialize model
finetune_model = SimpleNet().to(device)

# Track accuracy on all tasks after each training phase
accuracy_matrix = np.zeros((len(train_tasks), len(test_tasks)))

print("Training sequentially on 5 tasks...\n")

for current_task in range(len(train_tasks)):
    # Train on current task
    print(f"Training on Task {current_task} ({digit_pairs[current_task]})...")
    train_task(finetune_model, train_tasks[current_task], epochs=3)
    
    # Evaluate on all tasks seen so far
    for eval_task in range(len(test_tasks)):
        acc = evaluate_task(finetune_model, test_tasks[eval_task])
        accuracy_matrix[current_task, eval_task] = acc
        
        if eval_task <= current_task:
            print(f"  Task {eval_task}: {acc:.1f}%")
    print()

### Visualizing Catastrophic Forgetting

The accuracy matrix shows a clear pattern: as we learn new tasks (rows), performance on old tasks (earlier columns) degrades dramatically.

This is **catastrophic forgetting** in action.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of accuracy matrix
im = ax1.imshow(accuracy_matrix, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
ax1.set_xlabel('Task ID (evaluation)')
ax1.set_ylabel('Training Phase')
ax1.set_title('Accuracy Matrix: Catastrophic Forgetting')
ax1.set_xticks(range(len(test_tasks)))
ax1.set_yticks(range(len(train_tasks)))

# Add text annotations
for i in range(len(train_tasks)):
    for j in range(len(test_tasks)):
        text = ax1.text(j, i, f'{accuracy_matrix[i, j]:.0f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax1, label='Accuracy (%)')

# Line plot showing forgetting
for task_id in range(len(test_tasks)):
    # Plot accuracy on this task as we continue training on later tasks
    ax2.plot(range(task_id, len(train_tasks)), 
             accuracy_matrix[task_id:, task_id], 
             marker='o', label=f'Task {task_id}')

ax2.set_xlabel('Training Phase')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Forgetting Curves: Accuracy on Each Task Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate average forgetting
final_avg_acc = accuracy_matrix[-1, :].mean()
print(f"\nFinal average accuracy across all tasks: {final_avg_acc:.1f}%")

**Key observation:** After training on Task 4, the model performs well on Task 4 (>95%) but has almost completely forgotten Tasks 0-3 (often <20% accuracy).

This is the fundamental problem continual learning aims to solve.

## 5. Baseline Approaches

### 5.1 The Upper Bound: Joint Training

**Joint training** means training on all tasks simultaneously. This is the ideal scenario but often impractical:
- Requires storing all data from all tasks
- Not possible when tasks arrive sequentially
- Computationally expensive as dataset grows

However, it serves as our **upper bound** — the best possible performance.

In [ ]:
# Combine all training tasks
from torch.utils.data import ConcatDataset

joint_train_dataset = ConcatDataset(train_tasks)
print(f"Joint training dataset size: {len(joint_train_dataset)}")

# Train on all tasks together
joint_model = SimpleNet().to(device)
print("Training on all tasks jointly...")
train_task(joint_model, joint_train_dataset, epochs=5)

# Evaluate on all tasks
joint_accuracies = []
print("\nJoint training results:")
for task_id, test_task in enumerate(test_tasks):
    acc = evaluate_task(joint_model, test_task)
    joint_accuracies.append(acc)
    print(f"Task {task_id}: {acc:.1f}%")

print(f"\nAverage accuracy: {np.mean(joint_accuracies):.1f}%")

### 5.2 Task-Specific Output Heads

A simple approach: use **task-specific output layers** while sharing the feature extractor.

**Idea:** Each task gets its own classification head, but they share the same feature representations.

**Limitation:** The shared features still suffer from forgetting.

In [ ]:
class MultiHeadNet(nn.Module):
    """Network with shared features and task-specific heads."""
    
    def __init__(self, num_tasks=5, hidden_size=400):
        super().__init__()
        # Shared feature extractor
        self.fc1 = nn.Linear(28 * 28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
        # Task-specific heads (each outputs 10 classes)
        self.heads = nn.ModuleList([nn.Linear(hidden_size, 10) for _ in range(num_tasks)])
    
    def forward(self, x, task_id):
        # Shared features
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        
        # Task-specific prediction
        x = self.heads[task_id](x)
        return x

# This approach still suffers from forgetting in the shared layers
# We'll demonstrate this in the experiments section
print("Multi-head architecture defined")

## 6. Replay Methods: Remembering Through Examples

### Experience Replay

**Core idea:** Maintain a small buffer of examples from previous tasks. When learning a new task, interleave training on new data with replayed old examples.

**Advantages:**
- Simple and effective
- Works with any architecture
- Strong empirical performance

**Disadvantages:**
- Requires storing raw data (privacy/storage concerns)
- Performance depends on buffer size
- Not applicable when data cannot be stored

In [ ]:
class ReplayBuffer:
    """Fixed-size buffer for experience replay."""
    
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
    
    def add_samples(self, dataset, samples_per_task):
        """Add samples from a dataset to the buffer."""
        # Randomly sample from dataset
        indices = np.random.choice(len(dataset), 
                                   min(samples_per_task, len(dataset)), 
                                   replace=False)
        
        for idx in indices:
            if len(self.buffer) >= self.capacity:
                # Remove oldest sample if at capacity
                self.buffer.pop(0)
            self.buffer.append(dataset[int(idx)])
    
    def get_dataset(self):
        """Convert buffer to a PyTorch dataset."""
        return self.buffer
    
    def __len__(self):
        return len(self.buffer)

# Test the buffer
buffer = ReplayBuffer(capacity=500)
buffer.add_samples(train_tasks[0], samples_per_task=100)
print(f"Buffer size: {len(buffer)}")

### Training with Replay

We'll modify the training procedure to interleave new task data with replayed examples from the buffer.

In [ ]:
def train_task_with_replay(model, task_dataset, replay_buffer, epochs=3, batch_size=128, lr=0.001):
    """
    Train on current task while replaying examples from previous tasks.
    """
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Combine current task with replay buffer
    if len(replay_buffer) > 0:
        combined_dataset = ConcatDataset([task_dataset, replay_buffer.get_dataset()])
    else:
        combined_dataset = task_dataset
    
    loader = torch.utils.data.DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        epoch_loss = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(loader)

print("Replay training function ready")

Now let's train with experience replay and compare to the baseline.

In [ ]:
# Initialize model and replay buffer
replay_model = SimpleNet().to(device)
replay_buffer = ReplayBuffer(capacity=1000)  # Store 1000 examples total

# Track accuracy
replay_accuracy_matrix = np.zeros((len(train_tasks), len(test_tasks)))

print("Training with experience replay...\n")

for current_task in range(len(train_tasks)):
    print(f"Training on Task {current_task} ({digit_pairs[current_task]})...")
    
    # Train with replay
    train_task_with_replay(replay_model, train_tasks[current_task], replay_buffer, epochs=3)
    
    # Add samples from this task to buffer
    samples_per_task = 200  # Store 200 examples per task
    replay_buffer.add_samples(train_tasks[current_task], samples_per_task)
    print(f"  Buffer size: {len(replay_buffer)}")
    
    # Evaluate on all tasks
    for eval_task in range(len(test_tasks)):
        acc = evaluate_task(replay_model, test_tasks[eval_task])
        replay_accuracy_matrix[current_task, eval_task] = acc
        
        if eval_task <= current_task:
            print(f"  Task {eval_task}: {acc:.1f}%")
    print()

### Comparing Replay vs Fine-tuning

Let's visualize how experience replay reduces catastrophic forgetting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fine-tuning heatmap
im1 = axes[0].imshow(accuracy_matrix, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
axes[0].set_xlabel('Task ID (evaluation)')
axes[0].set_ylabel('Training Phase')
axes[0].set_title('Fine-tuning (Baseline)')
for i in range(len(train_tasks)):
    for j in range(len(test_tasks)):
        axes[0].text(j, i, f'{accuracy_matrix[i, j]:.0f}',
                    ha="center", va="center", color="black", fontsize=9)
plt.colorbar(im1, ax=axes[0], label='Accuracy (%)')

# Replay heatmap
im2 = axes[1].imshow(replay_accuracy_matrix, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
axes[1].set_xlabel('Task ID (evaluation)')
axes[1].set_ylabel('Training Phase')
axes[1].set_title('Experience Replay')
for i in range(len(train_tasks)):
    for j in range(len(test_tasks)):
        axes[1].text(j, i, f'{replay_accuracy_matrix[i, j]:.0f}',
                    ha="center", va="center", color="black", fontsize=9)
plt.colorbar(im2, ax=axes[1], label='Accuracy (%)')

# Comparison: final performance
final_finetune = accuracy_matrix[-1, :]
final_replay = replay_accuracy_matrix[-1, :]

x = np.arange(len(test_tasks))
width = 0.35
axes[2].bar(x - width/2, final_finetune, width, label='Fine-tuning', alpha=0.8)
axes[2].bar(x + width/2, final_replay, width, label='Replay', alpha=0.8)
axes[2].set_xlabel('Task ID')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Final Performance Comparison')
axes[2].set_xticks(x)
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nFine-tuning average: {final_finetune.mean():.1f}%")
print(f"Replay average: {final_replay.mean():.1f}%")
print(f"Improvement: +{final_replay.mean() - final_finetune.mean():.1f}%")

**Key insight:** Experience replay dramatically reduces forgetting. Notice how the earlier tasks (columns 0-3) maintain much higher accuracy in the replay heatmap.

The trade-off: we're storing 1000 examples (vs. 0 for fine-tuning). In some domains, this is acceptable. In others (privacy-sensitive data), we need different approaches.

## 7. Regularization Methods: Elastic Weight Consolidation (EWC)

### The Core Idea

**Elastic Weight Consolidation (EWC)** protects important weights from changing when learning new tasks.

**Intuition:** Some weights are critical for previous tasks, others are less important. EWC:
1. Identifies which weights are important for old tasks (using Fisher Information)
2. Adds a penalty term that resists changing these important weights
3. Allows unimportant weights to adapt freely to new tasks

**Mathematical formulation:**

$$\mathcal{L}(\theta) = \mathcal{L}_B(\theta) + \sum_i \frac{\lambda}{2} F_i (\theta_i - \theta^*_i)^2$$

Where:
- $\mathcal{L}_B(\theta)$ is the loss on task B (new task)
- $F_i$ is the Fisher Information for parameter $i$ (measures importance)
- $\theta^*_i$ is the optimal parameter value from task A (old task)
- $\lambda$ controls the regularization strength

### Computing Fisher Information

The **Fisher Information** measures how much a parameter affects the model's predictions. High Fisher Information means the parameter is important for the task.

We estimate it using the gradient of the log-likelihood:

$$F_i \approx \frac{1}{N} \sum_{n=1}^N \left(\frac{\partial \log p(y_n | x_n, \theta)}{\partial \theta_i}\right)^2$$

In [ ]:
def compute_fisher_information(model, dataset, num_samples=1000, batch_size=128):
    """
    Compute Fisher Information Matrix for all model parameters.
    
    Args:
        model: Neural network
        dataset: Dataset to compute Fisher over
        num_samples: Number of samples to use
    
    Returns:
        Dictionary mapping parameter names to Fisher Information values
    """
    model.eval()
    
    # Initialize Fisher Information to zeros
    fisher = {}
    for name, param in model.named_parameters():
        fisher[name] = torch.zeros_like(param)
    
    # Sample subset of data
    num_samples = min(num_samples, len(dataset))
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    subset = torch.utils.data.Subset(dataset, indices)
    loader = torch.utils.data.DataLoader(subset, batch_size=batch_size, shuffle=False)
    
    # Accumulate gradients
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        model.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        
        # Accumulate squared gradients (Fisher Information)
        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher[name] += param.grad.data ** 2
    
    # Normalize by number of samples
    for name in fisher:
        fisher[name] /= num_samples
    
    return fisher

print("Fisher Information computation ready")

Let's compute Fisher Information for a trained model and visualize which parameters are important.

In [ ]:
# Train a simple model on task 0
test_model = SimpleNet().to(device)
train_task(test_model, train_tasks[0], epochs=3)

# Compute Fisher Information
fisher = compute_fisher_information(test_model, train_tasks[0], num_samples=500)

# Visualize Fisher Information for each layer
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

layer_names = ['fc1.weight', 'fc2.weight', 'fc3.weight']
for idx, layer_name in enumerate(layer_names):
    fisher_values = fisher[layer_name].cpu().numpy().flatten()
    
    axes[idx].hist(np.log10(fisher_values + 1e-10), bins=50, alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel('log10(Fisher Information)')
    axes[idx].set_ylabel('Count')
    axes[idx].set_title(f'{layer_name}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFisher Information statistics:")
for layer_name in layer_names:
    fisher_values = fisher[layer_name].cpu().numpy().flatten()
    print(f"{layer_name}: mean={fisher_values.mean():.2e}, std={fisher_values.std():.2e}")

**Observation:** Fisher Information varies widely across parameters. Some are very important (high values), others less so (low values). EWC will protect the important ones.

### Implementing EWC Training

Now we'll implement the full EWC training procedure that adds the Fisher-weighted penalty to the loss.

In [ ]:
def ewc_loss(model, current_loss, fisher_dict, optimal_params, lambda_ewc=5000):
    """
    Compute EWC regularization loss.
    
    Args:
        model: Current model
        current_loss: Loss on current task
        fisher_dict: Fisher Information for previous task
        optimal_params: Optimal parameters from previous task
        lambda_ewc: Regularization strength
    
    Returns:
        Total loss with EWC penalty
    """
    ewc_penalty = 0
    
    for name, param in model.named_parameters():
        if name in fisher_dict:
            # Penalty = lambda/2 * Fisher * (theta - theta*)^2
            fisher = fisher_dict[name]
            optimal = optimal_params[name]
            ewc_penalty += (fisher * (param - optimal) ** 2).sum()
    
    total_loss = current_loss + (lambda_ewc / 2) * ewc_penalty
    return total_loss

def train_task_with_ewc(model, task_dataset, fisher_dict, optimal_params, 
                        epochs=3, batch_size=128, lr=0.001, lambda_ewc=5000):
    """
    Train on a task with EWC regularization.
    """
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(task_dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        epoch_loss = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            current_loss = F.cross_entropy(outputs, labels)
            
            # Add EWC penalty if we have previous task information
            if fisher_dict and optimal_params:
                total_loss = ewc_loss(model, current_loss, fisher_dict, optimal_params, lambda_ewc)
            else:
                total_loss = current_loss
            
            total_loss.backward()
            optimizer.step()
            
            epoch_loss += total_loss.item()
    
    return epoch_loss / len(loader)

print("EWC training functions ready")

### Training with EWC

Let's train sequentially using EWC and track performance.

In [ ]:
# Initialize model
ewc_model = SimpleNet().to(device)

# Track Fisher Information and optimal parameters for all previous tasks
all_fisher = {}
all_optimal_params = {}

# Merge Fisher Information from multiple tasks
def merge_fisher_dicts(fisher_list):
    """Combine Fisher Information from multiple tasks by summing."""
    if not fisher_list:
        return {}
    
    merged = {}
    for name in fisher_list[0].keys():
        merged[name] = sum(f[name] for f in fisher_list)
    return merged

# Track accuracy
ewc_accuracy_matrix = np.zeros((len(train_tasks), len(test_tasks)))

print("Training with EWC...\n")

fisher_list = []

for current_task in range(len(train_tasks)):
    print(f"Training on Task {current_task} ({digit_pairs[current_task]})...")
    
    # Merge Fisher from all previous tasks
    merged_fisher = merge_fisher_dicts(fisher_list)
    
    # Train with EWC
    train_task_with_ewc(ewc_model, train_tasks[current_task], 
                        merged_fisher, all_optimal_params, 
                        epochs=3, lambda_ewc=5000)
    
    # Compute Fisher for this task
    current_fisher = compute_fisher_information(ewc_model, train_tasks[current_task], num_samples=500)
    fisher_list.append(current_fisher)
    
    # Save optimal parameters
    all_optimal_params = {name: param.data.clone() 
                          for name, param in ewc_model.named_parameters()}
    
    # Evaluate on all tasks
    for eval_task in range(len(test_tasks)):
        acc = evaluate_task(ewc_model, test_tasks[eval_task])
        ewc_accuracy_matrix[current_task, eval_task] = acc
        
        if eval_task <= current_task:
            print(f"  Task {eval_task}: {acc:.1f}%")
    print()

### EWC vs Fine-tuning vs Replay

Let's compare all three approaches to see how EWC performs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot heatmaps
methods = [
    ('Fine-tuning', accuracy_matrix),
    ('Replay', replay_accuracy_matrix),
    ('EWC', ewc_accuracy_matrix)
]

for idx, (name, matrix) in enumerate(methods):
    row, col = idx // 2, idx % 2
    im = axes[row, col].imshow(matrix, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
    axes[row, col].set_xlabel('Task ID (evaluation)')
    axes[row, col].set_ylabel('Training Phase')
    axes[row, col].set_title(name)
    
    for i in range(len(train_tasks)):
        for j in range(len(test_tasks)):
            axes[row, col].text(j, i, f'{matrix[i, j]:.0f}',
                               ha="center", va="center", color="black", fontsize=8)
    
    plt.colorbar(im, ax=axes[row, col], label='Accuracy (%)')

# Final comparison bar plot
x = np.arange(len(test_tasks))
width = 0.25

axes[1, 1].bar(x - width, accuracy_matrix[-1, :], width, label='Fine-tuning', alpha=0.8)
axes[1, 1].bar(x, replay_accuracy_matrix[-1, :], width, label='Replay', alpha=0.8)
axes[1, 1].bar(x + width, ewc_accuracy_matrix[-1, :], width, label='EWC', alpha=0.8)

axes[1, 1].set_xlabel('Task ID')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].set_title('Final Performance Comparison')
axes[1, 1].set_xticks(x)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nAverage accuracy across all tasks (final):")
print(f"Fine-tuning: {accuracy_matrix[-1, :].mean():.1f}%")
print(f"Replay:      {replay_accuracy_matrix[-1, :].mean():.1f}%")
print(f"EWC:         {ewc_accuracy_matrix[-1, :].mean():.1f}%")
print(f"Joint (upper bound): {np.mean(joint_accuracies):.1f}%")

**Key observations:**

1. **EWC significantly reduces forgetting** compared to fine-tuning, though typically not as much as replay
2. **EWC requires no data storage** — it only stores Fisher Information and optimal parameters (same memory as the model)
3. **Performance depends on lambda** — the regularization strength trades off plasticity (learning new tasks) vs. stability (remembering old ones)

### Hyperparameter Sensitivity: Lambda in EWC

The parameter $\lambda$ controls the strength of EWC regularization. Let's explore how it affects the plasticity-stability trade-off.

In [ ]:
lambda_values = [0, 100, 1000, 5000, 20000]
lambda_results = {}

print("Testing different lambda values...\n")

for lambda_ewc in lambda_values:
    print(f"Lambda = {lambda_ewc}")
    
    # Initialize fresh model
    model = SimpleNet().to(device)
    fisher_list = []
    optimal_params = {}
    
    # Train on just the first two tasks (for speed)
    for task_id in range(2):
        merged_fisher = merge_fisher_dicts(fisher_list)
        train_task_with_ewc(model, train_tasks[task_id], 
                           merged_fisher, optimal_params, 
                           epochs=3, lambda_ewc=lambda_ewc)
        
        current_fisher = compute_fisher_information(model, train_tasks[task_id], num_samples=300)
        fisher_list.append(current_fisher)
        optimal_params = {name: param.data.clone() for name, param in model.named_parameters()}
    
    # Evaluate
    acc_task0 = evaluate_task(model, test_tasks[0])
    acc_task1 = evaluate_task(model, test_tasks[1])
    
    lambda_results[lambda_ewc] = (acc_task0, acc_task1)
    print(f"  Task 0: {acc_task0:.1f}%, Task 1: {acc_task1:.1f}%\n")

Visualize the plasticity-stability trade-off across different lambda values.

In [ ]:
lambdas = list(lambda_results.keys())
task0_accs = [lambda_results[l][0] for l in lambdas]
task1_accs = [lambda_results[l][1] for l in lambdas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Line plot
ax1.plot(lambdas, task0_accs, marker='o', label='Task 0 (old)', linewidth=2)
ax1.plot(lambdas, task1_accs, marker='s', label='Task 1 (new)', linewidth=2)
ax1.set_xlabel('Lambda (regularization strength)')
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Plasticity-Stability Trade-off')
ax1.set_xscale('symlog')  # Log scale to see both 0 and large values
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar plot
x = np.arange(len(lambdas))
width = 0.35
ax2.bar(x - width/2, task0_accs, width, label='Task 0 (old)', alpha=0.8)
ax2.bar(x + width/2, task1_accs, width, label='Task 1 (new)', alpha=0.8)
ax2.set_xlabel('Lambda')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Performance vs Regularization Strength')
ax2.set_xticks(x)
ax2.set_xticklabels([str(l) for l in lambdas])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Lambda = 0: No regularization = catastrophic forgetting")
print("- Low lambda: Good plasticity (learns Task 1 well) but some forgetting")
print("- High lambda: Strong stability (remembers Task 0) but reduced plasticity")
print("- Optimal lambda balances both, depends on task similarity and importance")

## 8. Parameter Isolation: Progressive Neural Networks

### The Idea: Separate Parameters Per Task

**Progressive Neural Networks** take a radical approach: allocate separate parameters for each task while allowing lateral connections to transfer knowledge.

**Advantages:**
- Zero forgetting (each task has dedicated capacity)
- Enables positive forward transfer (new tasks can use old features)
- No hyperparameter tuning for stability

**Disadvantages:**
- Model grows linearly with number of tasks
- Memory and computation scale poorly
- Not practical for hundreds of tasks

In [ ]:
class ProgressiveNet(nn.Module):
    """
    Progressive Neural Network: adds a new column for each task.
    Each column can use features from previous columns via lateral connections.
    """
    
    def __init__(self, input_size=784, hidden_size=200, output_size=10, device=None):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.device = device
        
        # Each task gets its own column (list of layers)
        self.columns = nn.ModuleList()
    
    def add_task(self):
        """Add a new column for a new task."""
        task_id = len(self.columns)
        
        # New column with lateral connections from previous columns
        # For simplicity, we use a single hidden layer per column
        if task_id == 0:
            # First task: standard network
            column = nn.ModuleDict({
                'fc1': nn.Linear(self.input_size, self.hidden_size),
                'fc2': nn.Linear(self.hidden_size, self.output_size)
            })
        else:
            # Later tasks: input from previous columns + own input
            lateral_size = self.hidden_size * task_id  # Features from all previous columns
            column = nn.ModuleDict({
                'fc1': nn.Linear(self.input_size + lateral_size, self.hidden_size),
                'fc2': nn.Linear(self.hidden_size, self.output_size)
            })
        
        # Move column to device if specified
        if self.device is not None:
            column = column.to(self.device)
        
        self.columns.append(column)
    
    def forward(self, x, task_id):
        """Forward pass using the specified task column."""
        x = x.view(x.size(0), -1)
        
        # Compute features from previous columns (frozen)
        prev_features = []
        with torch.no_grad():
            for i in range(task_id):
                if i == 0:
                    h = F.relu(self.columns[i]['fc1'](x))
                else:
                    # Concatenate with features from even earlier columns
                    prev_concat = torch.cat([x] + prev_features[:i], dim=1)
                    h = F.relu(self.columns[i]['fc1'](prev_concat))
                prev_features.append(h)
        
        # Current task column
        if task_id == 0:
            h = F.relu(self.columns[task_id]['fc1'](x))
        else:
            # Use lateral connections from previous tasks
            lateral = torch.cat(prev_features, dim=1)
            combined_input = torch.cat([x, lateral], dim=1)
            h = F.relu(self.columns[task_id]['fc1'](combined_input))
        
        output = self.columns[task_id]['fc2'](h)
        return output

print("Progressive Neural Network defined")

Let's train a Progressive Neural Network and verify zero forgetting.

In [ ]:
# Initialize Progressive Network
prog_net = ProgressiveNet(input_size=784, hidden_size=200, output_size=10, device=device).to(device)

# Track accuracy
prog_accuracy_matrix = np.zeros((len(train_tasks), len(test_tasks)))

print("Training Progressive Neural Network...\n")

for task_id in range(len(train_tasks)):
    print(f"Task {task_id} ({digit_pairs[task_id]})")
    
    # Add new column for this task
    prog_net.add_task()
    
    # Train only the new column (freeze previous columns)
    # Get parameters of the current column only
    current_params = list(prog_net.columns[task_id].parameters())
    
    optimizer = torch.optim.Adam(current_params, lr=0.001)
    loader = torch.utils.data.DataLoader(train_tasks[task_id], batch_size=128, shuffle=True)
    
    # Train for 3 epochs
    prog_net.train()
    for epoch in range(3):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = prog_net(images, task_id)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # Evaluate on all tasks
    prog_net.eval()
    for eval_task_id in range(len(test_tasks)):
        if eval_task_id <= task_id:  # Only evaluate tasks we've learned
            eval_loader = torch.utils.data.DataLoader(test_tasks[eval_task_id], 
                                                      batch_size=128, shuffle=False)
            correct = 0
            total = 0
            
            with torch.no_grad():
                for images, labels in eval_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = prog_net(images, eval_task_id)
                    _, predicted = outputs.max(1)
                    total += labels.size(0)
                    correct += predicted.eq(labels).sum().item()
            
            acc = 100.0 * correct / total
            prog_accuracy_matrix[task_id, eval_task_id] = acc
            print(f"  Task {eval_task_id}: {acc:.1f}%")
    print()

# Count parameters
total_params = sum(p.numel() for p in prog_net.parameters())
print(f"Total parameters: {total_params:,}")

### Progressive Networks: Zero Forgetting at a Cost

Let's visualize the Progressive Network's performance and compare it to other methods.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Progressive Network heatmap
im = axes[0].imshow(prog_accuracy_matrix, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
axes[0].set_xlabel('Task ID (evaluation)')
axes[0].set_ylabel('Training Phase')
axes[0].set_title('Progressive Neural Network')

for i in range(len(train_tasks)):
    for j in range(len(test_tasks)):
        if j <= i:  # Only show evaluated tasks
            axes[0].text(j, i, f'{prog_accuracy_matrix[i, j]:.0f}',
                        ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=axes[0], label='Accuracy (%)')

# Compare all methods
methods_comparison = {
    'Fine-tune': accuracy_matrix[-1, :],
    'Replay': replay_accuracy_matrix[-1, :],
    'EWC': ewc_accuracy_matrix[-1, :],
    'Progressive': prog_accuracy_matrix[-1, :],
    'Joint': joint_accuracies
}

x = np.arange(len(test_tasks))
width = 0.15
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

for i, (name, accs) in enumerate(methods_comparison.items()):
    offset = (i - 2) * width
    axes[1].bar(x + offset, accs, width, label=name, alpha=0.8, color=colors[i])

axes[1].set_xlabel('Task ID')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Final Performance: All Methods')
axes[1].set_xticks(x)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nFinal average accuracy:")
for name, accs in methods_comparison.items():
    print(f"{name:12s}: {np.mean(accs):.1f}%")

print("\nKey insight: Progressive Networks achieve zero forgetting (notice the diagonal)")
print("but at the cost of growing model size.")

**Observation:** Progressive Networks maintain perfect (or near-perfect) accuracy on all previous tasks. The diagonal shows no degradation over time.

The trade-off: model size grows with each task, making this impractical for long task sequences.

## 9. Continual Learning Evaluation Metrics

### Beyond Average Accuracy

Average accuracy doesn't tell the full story. We need metrics that capture:
1. **Forgetting**: How much performance degrades on old tasks
2. **Forward transfer**: Does learning task A help with task B?
3. **Backward transfer**: Does learning task B hurt task A?

Let's implement standard continual learning metrics.

In [ ]:
def compute_forgetting(accuracy_matrix):
    """
    Compute forgetting metric: how much accuracy dropped from peak to final.
    
    Forgetting_i = max_j(Acc_i,j) - Acc_i,T
    
    Where Acc_i,j is accuracy on task i after training on task j,
    and T is the final training phase.
    """
    num_tasks = accuracy_matrix.shape[1]
    forgetting = []
    
    for task_id in range(num_tasks - 1):  # Exclude last task (no time to forget)
        # Find peak accuracy on this task
        peak_acc = accuracy_matrix[task_id:, task_id].max()
        # Final accuracy on this task
        final_acc = accuracy_matrix[-1, task_id]
        # Forgetting
        forgetting.append(peak_acc - final_acc)
    
    return np.mean(forgetting)

def compute_forward_transfer(accuracy_matrix, random_baseline):
    """
    Compute forward transfer: does learning previous tasks help with new ones?
    
    FWT = (1/T) * sum_i (Acc_i,i - Random_i)
    
    Positive FWT means prior knowledge helps.
    """
    num_tasks = accuracy_matrix.shape[1]
    transfer = []
    
    for task_id in range(num_tasks):
        # Accuracy when first learning this task
        initial_acc = accuracy_matrix[task_id, task_id]
        # Compare to random baseline
        transfer.append(initial_acc - random_baseline)
    
    return np.mean(transfer)

def compute_backward_transfer(accuracy_matrix):
    """
    Compute backward transfer: does learning new tasks help old ones?
    
    BWT = (1/T-1) * sum_{i<T} (Acc_i,T - Acc_i,i)
    
    Positive BWT is rare (usually negative due to forgetting).
    """
    num_tasks = accuracy_matrix.shape[1]
    transfer = []
    
    for task_id in range(num_tasks - 1):
        # Accuracy when first learned
        initial_acc = accuracy_matrix[task_id, task_id]
        # Accuracy after all training
        final_acc = accuracy_matrix[-1, task_id]
        # Transfer (usually negative)
        transfer.append(final_acc - initial_acc)
    
    return np.mean(transfer)

print("Evaluation metrics defined")

Let's compute these metrics for all our methods.

In [ ]:
# Random baseline (50% for binary classification)
random_baseline = 50.0

methods_matrices = {
    'Fine-tuning': accuracy_matrix,
    'Replay': replay_accuracy_matrix,
    'EWC': ewc_accuracy_matrix,
    'Progressive': prog_accuracy_matrix
}

results_table = []

print("Continual Learning Metrics:\n")
print(f"{'Method':<15} {'Avg Acc':<10} {'Forgetting':<12} {'Fwd Transfer':<12} {'Bwd Transfer':<12}")
print("-" * 65)

for method_name, matrix in methods_matrices.items():
    avg_acc = matrix[-1, :].mean()
    forgetting = compute_forgetting(matrix)
    fwd_transfer = compute_forward_transfer(matrix, random_baseline)
    bwd_transfer = compute_backward_transfer(matrix)
    
    print(f"{method_name:<15} {avg_acc:<10.1f} {forgetting:<12.1f} {fwd_transfer:<12.1f} {bwd_transfer:<12.1f}")
    
    results_table.append({
        'method': method_name,
        'avg_acc': avg_acc,
        'forgetting': forgetting,
        'fwd_transfer': fwd_transfer,
        'bwd_transfer': bwd_transfer
    })

print("\nInterpretation:")
print("- Forgetting: Lower is better (0 = no forgetting)")
print("- Forward Transfer: Higher is better (positive = prior knowledge helps)")
print("- Backward Transfer: Higher is better (positive = new learning helps old tasks)")

Visualize the metrics comparison across methods.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['avg_acc', 'forgetting', 'bwd_transfer']
titles = ['Average Accuracy', 'Forgetting (lower better)', 'Backward Transfer']
colors_map = {'Fine-tuning': '#d62728', 'Replay': '#ff7f0e', 'EWC': '#2ca02c', 'Progressive': '#1f77b4'}

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    methods = [r['method'] for r in results_table]
    values = [r[metric] for r in results_table]
    colors = [colors_map[m] for m in methods]
    
    bars = axes[idx].bar(range(len(methods)), values, color=colors, alpha=0.8)
    axes[idx].set_xticks(range(len(methods)))
    axes[idx].set_xticklabels(methods, rotation=15, ha='right')
    axes[idx].set_ylabel('Value')
    axes[idx].set_title(title)
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{height:.1f}',
                      ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 10. Key Takeaways and Practical Guidance

### Summary of Approaches

We explored three major families of continual learning methods:

**1. Replay Methods (Experience Replay)**
- Store examples from previous tasks
- Interleave old and new data during training
- Pros: Simple, effective, works with any architecture
- Cons: Requires data storage, privacy concerns
- Best for: When data can be stored, strong empirical performance needed

**2. Regularization Methods (EWC)**
- Protect important parameters from changing
- Use Fisher Information to identify importance
- Pros: No data storage, theoretically grounded
- Cons: Requires hyperparameter tuning (lambda), moderate performance
- Best for: Privacy-sensitive applications, when storage is limited

**3. Parameter Isolation (Progressive Networks)**
- Allocate separate parameters per task
- Enable knowledge transfer via lateral connections
- Pros: Zero forgetting, positive transfer
- Cons: Model grows linearly, not scalable
- Best for: Small number of high-value tasks, when forgetting is unacceptable

### When to Use Each Approach

**Choose Replay if:**
- You can store data
- Privacy is not a primary concern
- You need the best empirical performance
- Tasks are diverse and complex

**Choose EWC if:**
- Data cannot be stored (privacy, regulations)
- Memory is constrained
- You're willing to tune hyperparameters
- Tasks are moderately similar

**Choose Progressive Networks if:**
- You have very few tasks (< 10)
- Zero forgetting is critical
- Model size is not a constraint
- You want to study transfer learning

**Choose Joint Training if:**
- All data is available simultaneously
- Computational resources are abundant
- You need the absolute best performance

### Open Problems and Future Directions

Continual learning remains an active research area. Key challenges:

**1. Task-incremental vs Class-incremental vs Domain-incremental Learning**
- We studied task-incremental (task ID known at test time)
- Class-incremental (task ID unknown) is much harder
- Domain-incremental (same task, shifting data) is common in practice

**2. Scalability**
- Methods tested on 5-20 tasks in research
- Real systems may face hundreds or thousands of tasks
- Need better memory-computation trade-offs

**3. Online Learning**
- We assumed clear task boundaries
- Real data streams are continuous and unlabeled
- Need methods that work without task labels

**4. Catastrophic Forgetting in LLMs**
- Large language models face the same issues
- RLHF and instruction tuning can cause forgetting
- Active area of research (LoRA, prefix tuning, etc.)

**5. Meta-learning for Continual Learning**
- Can we learn how to learn continually?
- Meta-learning + continual learning is a promising direction

### Practical Recommendations

For production systems:

1. **Start simple**: Try fine-tuning + small replay buffer first
2. **Monitor forgetting**: Track performance on validation sets from all tasks
3. **Hybrid approaches**: Combine methods (e.g., EWC + small replay buffer)
4. **Task relatedness matters**: Similar tasks forget less, diverse tasks forget more
5. **Consider architecture**: 
   - Modular designs (adapters, LoRA) reduce interference
   - Larger models forget less (more capacity)
6. **Budget for retraining**: Sometimes joint training on a schedule is simpler than continual learning

### Final Visualization: The Trade-off Space

In [ ]:
# Create a trade-off comparison
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 8))

# Define method characteristics (approximate scores 0-10)
methods_tradeoffs = {
    'Fine-tuning': {'performance': 2, 'memory': 10, 'computation': 10, 'scalability': 10},
    'Replay': {'performance': 9, 'memory': 5, 'computation': 7, 'scalability': 7},
    'EWC': {'performance': 6, 'memory': 9, 'computation': 8, 'scalability': 9},
    'Progressive': {'performance': 10, 'memory': 3, 'computation': 4, 'scalability': 2},
    'Joint': {'performance': 10, 'memory': 1, 'computation': 1, 'scalability': 1}
}

# Prepare data
categories = ['Performance', 'Memory\nEfficiency', 'Computation\nEfficiency', 'Scalability']
method_names = list(methods_tradeoffs.keys())
colors_list = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

# Create radar chart
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

ax = plt.subplot(111, projection='polar')

for idx, (method, scores) in enumerate(methods_tradeoffs.items()):
    values = [scores['performance'], scores['memory'], scores['computation'], scores['scalability']]
    values += values[:1]  # Complete the circle
    
    ax.plot(angles, values, 'o-', linewidth=2, label=method, color=colors_list[idx])
    ax.fill(angles, values, alpha=0.15, color=colors_list[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2', '4', '6', '8', '10'], size=9)
ax.grid(True, alpha=0.3)
ax.set_title('Continual Learning Methods: Trade-off Comparison\n(Higher = Better)', 
             size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

print("\nTrade-off Summary:")
print("- Fine-tuning: Fast but forgets")
print("- Replay: Strong performance, moderate cost")
print("- EWC: Good balance, no data storage")
print("- Progressive: Best performance, worst scalability")
print("- Joint: Optimal performance, impractical for continual setting")

### What We've Learned

In this notebook, we've built a comprehensive understanding of continual learning:

1. **The Problem**: Catastrophic forgetting prevents neural networks from learning sequentially
2. **The Measurement**: Proper metrics (forgetting, transfer) reveal the full picture
3. **The Solutions**: 
   - Replay: Remember through examples
   - Regularization: Protect important weights
   - Isolation: Separate parameters per task
4. **The Trade-offs**: No perfect solution — choose based on constraints
5. **The Practice**: Real systems need hybrid approaches and continuous monitoring

**The future of AI depends on continual learning.** As we move toward lifelong learning agents that can adapt without forgetting, these techniques become increasingly critical.

Whether you're fine-tuning language models, updating recommendation systems, or building robots that learn from experience, understanding catastrophic forgetting and how to prevent it is essential.